In [ ]:
from google.colab import files
import os

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:26<00:00, 207MB/s]



In [ ]:
!unzip -q skin-cancer-mnist-ham10000.zip
print("Dataset Downloaded and Unzipped Successfully!")

Dataset Downloaded and Unzipped Successfully!


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from glob import glob
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
!pip install imbalanced-learn
from imblearn.over_sampling import RandomOverSampler
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/HAM10000_metadata.csv')
image_path = {os.path.splitext(os.path.basename(x))[0]: x for x in glob(os.path.join('/content/', '*', '*.jpg'))}
df['path'] = df['image_id'].map(image_path)
df = df[df['path'].notna()]
X = df.drop(['dx'], axis=1)
y = df['dx']

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)

df_balanced = pd.concat([X_resampled, y_resampled], axis=1)

df_balanced['cell_type'] = df_balanced['dx'].map({
    'nv': 'Melanocytic nevi', 'mel': 'Melanoma', 'bkl': 'Benign keratosis',
    'bcc': 'Basal cell carcinoma', 'akiec': 'Actinic keratoses',
    'vasc': 'Vascular lesions', 'df': 'Dermatofibroma'
})

train_df, test_df = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['dx'])

print("Balanced Dataset Class Counts:")
print(df_balanced['dx'].value_counts())

Balanced Dataset Class Counts:
dx
bkl      6705
nv       6705
df       6705
mel      6705
vasc     6705
bcc      6705
akiec    6705
Name: count, dtype: int64


In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_loader = train_datagen.flow_from_dataframe(
    dataframe=train_df, x_col='path', y_col='cell_type',
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)

test_loader = test_datagen.flow_from_dataframe(
    dataframe=test_df, x_col='path', y_col='cell_type',
    target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)

Found 37548 validated image filenames belonging to 7 classes.
Found 9387 validated image filenames belonging to 7 classes.


In [ ]:
from tensorflow.keras.applications import ResNet50V2
base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(7, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

history = model.fit(
    train_loader,
    validation_data=test_loader,
    epochs=15,
    callbacks=[lr_scheduler]
)

Epoch 1/15
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 747s 625ms/step - accuracy: 0.6616 - loss: 0.9094 - val_accuracy: 0.7475 - val_loss: 0.6669 - learning_rate: 0.0010
Epoch 2/15
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 727s 619ms/step - accuracy: 0.7636 - loss: 0.6348 - val_accuracy: 0.8138 - val_loss: 0.5026 - learning_rate: 0.0010
Epoch 3/15
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 716s 610ms/step - accuracy: 0.7995 - loss: 0.5376 - val_accuracy: 0.8581 - val_loss: 0.4192 - learning_rate: 0.0010
Epoch 4/15
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 719s 613ms/step - accuracy: 0.8176 - loss: 0.4822 - val_accuracy: 0.8586 - val_loss: 0.3770 - learning_rate: 0.0010
Epoch 5/15
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 709s 604ms/step - accuracy: 0.8358 - loss: 0.4310 - val_accuracy: 0.8767 - val_loss: 0.3496 - learning_rate: 0.0010
Epoch 6/15
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 704s 600ms/step - accuracy: 0.8464 - loss: 0.4106 - val_accuracy: 0.8747 - val_loss: 0.3511 - learning_rate: 0.0010
Epoch 7/15
1174/1174 ━━━━━━━━━━━━━━━━━━━━ 721s 614ms

In [ ]:
model.save('skin_disease.keras')
from google.colab import files
files.download('skin_disease.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

In [ ]:
model = tf.keras.models.load_model('skin_disease.keras')
print('Model loaded successfully!')

Model loaded successfully!


In [ ]:
class_labels = {
    0: 'Actinic keratoses',
    1: 'Basal cell carcinoma',
    2: 'Benign keratosis',
    3: 'Dermatofibroma',
    4: 'Melanocytic nevi',
    5: 'Melanoma',
    6: 'Vascular lesions'
}

def predict_image(image_input):
    img = Image.fromarray(image_input.astype('uint8'), 'RGB')
    img = img.resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)[0]

    confidences = {class_labels[i]: float(predictions[i]) for i in range(len(class_labels))}
    return confidences

In [ ]:
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type='numpy', label='Upload Skin Lesion Image'),
    outputs=gr.Label(num_top_classes=7),
    title='Skin Lesion Classifier',
    description='Upload an image of a skin lesion to get a classification of its type.'
)

interface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5b9c50f2f3a1c34e09.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_4228/2296590634.py:13: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(image_input.astype('uint8'), 'RGB')


1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step


/tmp/ipykernel_4228/2296590634.py:13: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(image_input.astype('uint8'), 'RGB')
